In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from classification_training_reporter import TrainingReporter
from classification_svc_model import build_model_from_grid_params
from classification_dataset_preprocessing import make_preprocessing_pipeline, make_label_pipeline, make_training_pipeline, make_delta_pipeline, make_standarize_pipeline
from classification_svc_model import CustomSVCModel


# Odczyt datasetu

In [2]:
df = pd.read_csv(os.path.join("../data", "ortodoncja.csv"))

# Podział danych na zbiór treningowy i testowy

In [3]:
# pipeline = make_label_pipeline()
# df_processed = pipeline.fit_transform(df)
# label_encoder = pipeline.named_steps["encode_labels"].encoder
#
# X = df_processed.drop(columns=["growth direction"])
# y = df_processed["growth direction"]
#
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#
# tpipeline = make_training_pipeline()
# X_train, y_train = tpipeline.fit_resample(X_train,y_train)
# X_test = tpipeline[:1].fit_transform(X_test)

pipeline = make_label_pipeline()
df_processed = pipeline.fit_transform(df)
label_encoder = pipeline.named_steps["encode_labels"].encoder

X = df_processed.drop(columns=["growth direction"])
y = df_processed["growth direction"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

spipeline = make_standarize_pipeline()
X_train = spipeline.fit_transform(X_train)
X_test = spipeline.fit_transform(X_test)

# Inicjalizacja reportera do treningów

In [4]:
model = CustomSVCModel()
reporter = TrainingReporter(model, X_train, X_test, y_train, y_test)

# Pierwszy trening

In [5]:
reporter.train()

Start training...
Training finished!
Train Accuracy: 0.8315  |  Test Accuracy: 0.8000
Train F1:       0.8330  |  Test F1:       0.7985
Train AUROC:    0.9377  |  Test AUROC:    0.8900
---------------------------------------------------


# Cross walidacja

In [6]:
reporter.run_cross_validation(cv=10)

Start cross validation...
Fold 0:
  Train Accuracy: 0.8438  |  Val Accuracy: 0.6944
  Train F1:       0.8453  |  Val F1:       0.6954
---------------------------------------------------
Fold 1:
  Train Accuracy: 0.8313  |  Val Accuracy: 0.8333
  Train F1:       0.8319  |  Val F1:       0.8350
---------------------------------------------------
Fold 2:
  Train Accuracy: 0.8500  |  Val Accuracy: 0.8611
  Train F1:       0.8507  |  Val F1:       0.8481
---------------------------------------------------
Fold 3:
  Train Accuracy: 0.8375  |  Val Accuracy: 0.8056
  Train F1:       0.8380  |  Val F1:       0.7940
---------------------------------------------------
Fold 4:
  Train Accuracy: 0.8344  |  Val Accuracy: 0.7778
  Train F1:       0.8355  |  Val F1:       0.7845
---------------------------------------------------
Fold 5:
  Train Accuracy: 0.8594  |  Val Accuracy: 0.5556
  Train F1:       0.8622  |  Val F1:       0.5503
---------------------------------------------------
Fold 6:
  Trai

# Randomized grid search

In [7]:
random_grid = reporter.run_randomized_search_svc(cv=5)

Start randomized grid search for SVC...
Randomized search finished!
Best params: {'kernel': 'rbf', 'gamma': 0.01, 'degree': 2, 'C': 10.0}
Best F1 score: 0.7471924510218477
---------------------------------------------------


# Kroswalidacja po dostosowaniu hiperparametrów za pomocą Randomized Grid Search

In [8]:
model_RGS = build_model_from_grid_params(random_grid.best_params_)
reporter_RGS = TrainingReporter(model_RGS, X_train, X_test, y_train, y_test)
reporter_RGS.run_cross_validation(cv=10)

Start cross validation...
Fold 0:
  Train Accuracy: 0.8406  |  Val Accuracy: 0.6944
  Train F1:       0.8416  |  Val F1:       0.6954
---------------------------------------------------
Fold 1:
  Train Accuracy: 0.8313  |  Val Accuracy: 0.8889
  Train F1:       0.8318  |  Val F1:       0.8919
---------------------------------------------------
Fold 2:
  Train Accuracy: 0.8594  |  Val Accuracy: 0.8333
  Train F1:       0.8591  |  Val F1:       0.8193
---------------------------------------------------
Fold 3:
  Train Accuracy: 0.8406  |  Val Accuracy: 0.7222
  Train F1:       0.8407  |  Val F1:       0.7124
---------------------------------------------------
Fold 4:
  Train Accuracy: 0.8313  |  Val Accuracy: 0.7500
  Train F1:       0.8315  |  Val F1:       0.7569
---------------------------------------------------
Fold 5:
  Train Accuracy: 0.8594  |  Val Accuracy: 0.5833
  Train F1:       0.8601  |  Val F1:       0.5625
---------------------------------------------------
Fold 6:
  Trai

# Poważna niestabilność